In [21]:

import cmdstanpy
import numpy as np
import pandas as pd 
from scipy.special import expit  # inverse logit
import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel
from joblib import Parallel, delayed

# ---------------------------------------------------------------------
# Assumes df has: participant_id (1..P), sat_int (1..3), rt, correct
# ---------------------------------------------------------------------


In [32]:
df = pd.read_csv("../forstmann.csv")

df["correct"] = df.apply(lambda row: 1 if row["S"] == row["R"] else 0,axis = 1)
df['participant_id'], participant_labels = pd.factorize(df['subjects'])
df['participant_id'] += 1  
df = df[df["participant_id"]==2]

sat_map = {"speed":1,"neutral":2,"accuracy":3}
df["sat_int"] = df["E"].map(sat_map)
df["pc"] = (df["participant_id"]-1)*3 + df["sat_int"]
df

,subjects,E,S,R,rt,correct,participant_id,sat_int,pc
810,bd6t,speed,right,right,0.3240,1,2,1,4
811,bd6t,neutral,left,left,0.3974,1,2,2,5
812,bd6t,neutral,left,left,0.4014,1,2,2,5
813,bd6t,neutral,left,left,0.4131,1,2,2,5
814,bd6t,neutral,right,right,0.3381,1,2,2,5
...,...,...,...,...,...,...,...,...,...
1654,bd6t,speed,right,left,0.3433,0,2,1,4
1655,bd6t,accuracy,right,right,0.7795,1,2,3,6
1656,bd6t,speed,right,right,0.4021,1,2,1,4
1657,bd6t,neutral,right,right,0.6772,1,2,2,5


In [33]:
"""
Fit four rival single-subject models (c/B, c/t0, d'/B, d'/t0 free per
condition) to every participant, in parallel.

Saves THREE things per (participant, model) fit, so nothing has to be
re-run later:
  1. per_subject_fits_{model}.csv   - parameter estimates, WITH labels
  2. diagnostics_{model}.csv        - one row per participant: divergences,
                                       treedepth, max Rhat, min ESS
  3. idata/{model}_p{pid}.nc        - full posterior + log_lik, for LOO
"""

import traceback
from pathlib import Path

import numpy as np
import pandas as pd

import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

import arviz as az  # everything else, imported after

import arviz as az
from cmdstanpy import CmdStanModel
from joblib import Parallel, delayed

COND_LEVELS = [1, 2, 3]  # sat_int levels: speed / neutral / accuracy


# ---------------------------------------------------------------------------
# 1. Define the four models to compare.
# ---------------------------------------------------------------------------

def init_c_B(n_cond):
    return {'c': [0.0]*n_cond, 'B': [1.0]*n_cond, 't0': 0.2, 'd': 1.5, 'r': 4.5, 'p_lapse': 0.02}

def init_c_t0(n_cond):
    return {'c': [0.0]*n_cond, 't0': [0.2]*n_cond, 'd': 1.5, 'r': 4.5, 'B': 1.0, 'p_lapse': 0.02}

def init_d_B(n_cond):
    return {'d': [1.5]*n_cond, 'B': [1.0]*n_cond, 'c': 0.0, 't0': 0.2, 'p_lapse': 0.02}

def init_d_t0(n_cond):
    return {'d': [1.5]*n_cond, 't0': [0.2]*n_cond, 'c': 0.0, 'B': 1.0, 'r': 4.5, 'p_lapse': 0.02}

MODEL_SPECS = [
    {'name': 'c_B',  'stan_file': 'SRDM_single_c_B.stan',  'init_fn': init_c_B,  't0_varies': False},
    {'name': 'c_t0', 'stan_file': 'SRDM_single_c_t0.stan', 'init_fn': init_c_t0, 't0_varies': True},
    {'name': 'd_B',  'stan_file': 'SRDM_single_d_B.stan',  'init_fn': init_d_B,  't0_varies': False},
    {'name': 'd_t0', 'stan_file': 'SRDM_single_d_t0.stan', 'init_fn': init_d_t0, 't0_varies': True},
]

for spec in MODEL_SPECS:
    spec['model'] = CmdStanModel(stan_file=spec['stan_file'])


# ---------------------------------------------------------------------------
# 2. Data prep (unchanged).
# ---------------------------------------------------------------------------

def build_data(d, t0_varies):
    d_correct = d[d['correct'] == 1]
    d_false = d[d['correct'] == 0]

    max_rt = d.groupby('sat_int')['rt'].max().reindex(COND_LEVELS)
    if max_rt.isna().any():
        raise ValueError("empty condition cell (max_rt)")

    if t0_varies:
        t0_hi = d.groupby('sat_int')['rt'].quantile(0.05).reindex(COND_LEVELS)
        if t0_hi.isna().any():
            raise ValueError("empty condition cell (t0_hi)")
        t0_hi_val = t0_hi.to_numpy()
    else:
        t0_hi_val = float(d['rt'].quantile(0.05))

    return {
        'N_correct': len(d_correct), 'N_false': len(d_false),
        'rt_correct': d_correct['rt'].to_numpy(), 'rt_false': d_false['rt'].to_numpy(),
        'cond_correct': d_correct['sat_int'].to_numpy(dtype=int),
        'cond_false': d_false['sat_int'].to_numpy(dtype=int),
        'max_rt': max_rt.to_numpy(), 't0_hi': t0_hi_val,
    }


# ---------------------------------------------------------------------------
# 3. Per-participant, per-model fit function - now returns 3 pieces.
# ---------------------------------------------------------------------------

def fit_one(pid, df, spec, idata_dir):
    try:
        d = df[df['participant_id'] == pid]
        data = build_data(d, spec['t0_varies'])
        init_values = spec['init_fn'](len(COND_LEVELS))

        fit = spec['model'].sample(
            data=data, chains=4, parallel_chains=1,
            inits=init_values, iter_warmup=1000, iter_sampling=1000,
            adapt_delta=0.95, show_progress=False,
        )

        # --- 1. parameter table, WITH labels preserved this time ---
        summary = fit.summary().reset_index().rename(columns={'index': 'param'})
        summary['participant_id'] = pid
        summary['model'] = spec['name']

        # --- 2. diagnostics: divergences, treedepth, worst Rhat/ESS ---
        sv = fit.method_variables()
        n_iter_total = sv['divergent__'].size
        diagnostics = {
            'participant_id': pid,
            'model': spec['name'],
            'n_divergent': int(sv['divergent__'].sum()),
            'pct_divergent': float(sv['divergent__'].sum()) / n_iter_total,
            'mean_treedepth': float(np.mean(sv['treedepth__'])),
            'max_treedepth': float(np.max(sv['treedepth__'])),
            'mean_n_leapfrog': float(np.mean(sv['n_leapfrog__'])),
            'max_rhat': float(summary['R_hat'].max()),
            'min_ess_bulk': float(summary['ESS_bulk'].min()),
        }

        # --- 3. full posterior + log_lik, for LOO later ---
        idata = az.from_cmdstanpy(fit, log_likelihood='log_lik')
        idata.to_netcdf(idata_dir / f"{spec['name']}_p{pid}.nc")

        return summary, diagnostics

    except Exception as exc:
        print(f"[FAILED] participant={pid} model={spec['name']}: {exc}")
        traceback.print_exc()
        return None, None


# ---------------------------------------------------------------------------
# 4. Run: outer loop over models, inner parallel loop over participants.
# ---------------------------------------------------------------------------

def run_all(df, out_dir='.'):
    out_dir = Path(out_dir)
    idata_dir = out_dir / 'idata'
    out_dir.mkdir(parents=True, exist_ok=True)
    idata_dir.mkdir(parents=True, exist_ok=True)

    participants = sorted(df['participant_id'].unique())
    all_summaries, all_diagnostics = [], []

    for spec in MODEL_SPECS:
        print(f"=== Fitting model: {spec['name']} ({len(participants)} participants) ===")

        results = Parallel(n_jobs=-1)(
            delayed(fit_one)(pid, df, spec, idata_dir) for pid in participants
        )
        summaries = [s for s, _ in results if s is not None]
        diags = [d for _, d in results if d is not None]

        if summaries:
            model_df = pd.concat(summaries)
            model_df.to_csv(out_dir / f"per_subject_fits_{spec['name']}.csv", index=False)
            all_summaries.append(model_df)

        if diags:
            diag_df = pd.DataFrame(diags)
            diag_df.to_csv(out_dir / f"diagnostics_{spec['name']}.csv", index=False)
            all_diagnostics.append(diag_df)

            n_bad = (diag_df['pct_divergent'] > 0.01).sum()
            print(f"  -> {n_bad}/{len(diag_df)} participants had >1% divergent transitions")
        else:
            print(f"  !! no successful fits for model {spec['name']}")

    combined_summary = pd.concat(all_summaries) if all_summaries else pd.DataFrame()
    combined_diag = pd.concat(all_diagnostics) if all_diagnostics else pd.DataFrame()

    combined_summary.to_csv(out_dir / 'per_subject_fits_all_models.csv', index=False)
    combined_diag.to_csv(out_dir / 'diagnostics_all_models.csv', index=False)

    return combined_summary, combined_diag


if __name__ == '__main__':
    
    all_results, all_diagnostics = run_all(df, out_dir='fits')
    pass

=== Fitting model: c_B (1 participants) ===


13:34:02 - cmdstanpy - INFO - CmdStan start processing
13:34:02 - cmdstanpy - INFO - Chain [1] start processing
13:34:04 - cmdstanpy - INFO - Chain [1] done processing
13:34:04 - cmdstanpy - ERROR - Chain [1] error: terminated by signal 2 Unknown error: -2
13:34:04 - cmdstanpy - INFO - Chain [2] start processing


KeyboardInterrupt: 

In [34]:
import arviz as az
import pandas as pd

models = ['c_B', 'c_t0', 'd_B', 'd_t0']
participants = range(1, 20)

# per-participant comparison + tally of which model wins for each person
wins = {m: 0 for m in models}
per_participant_results = []

for pid in participants:
    idatas = {}
    for m in models:
        try:
            idatas[m] = az.from_netcdf(f'fits/idata/{m}_p{pid}.nc')
        except FileNotFoundError:
            continue  # skip if a fit failed to save

    if len(idatas) < 2:
        continue

    comp = az.compare(idatas)
    top_model = comp.index[0]
    wins[top_model] += 1

    comp['participant_id'] = pid
    per_participant_results.append(comp.reset_index())

print("Win counts across participants:")
print(wins)

all_comparisons = pd.concat(per_participant_results)
all_comparisons.to_csv('loo_comparison_all_participants.csv', index=False)

Win counts across participants:
{'c_B': 3, 'c_t0': 8, 'd_B': 4, 'd_t0': 4}
